In [1]:
from nb_utils import set_root

PROJECT_ROOT = set_root(level = 2)

# Supervised Method Evaluation with Toy Problems

This notebook opens the workshop with a simple claim: two models can have similar headline metrics while behaving very differently on easy and hard examples.

The goal is not to build the best classifier. The goal is to understand what standard evaluation reveals, what it hides, and why that motivates a latent-ability perspective.

## Learning Objectives

By the end of this notebook, participants should be able to:

- explain why accuracy alone is often insufficient;
- compare balanced and imbalanced settings;
- reason about the role of overlap and noise in example difficulty;
- identify examples that remain consistently hard;
- motivate the move from supervised evaluation to latent ability.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd

from utils.handson import (
    evaluate_logistic_workflow,
    make_toy_classification_dataset,
    plot_classification_dataset,
    summarize_classification_results,
)

## Step 1: Start with a friendly dataset

We first build a relatively easy binary classification problem.
The classes are reasonably separated and approximately balanced.

This is useful for the audience because it gives an immediate visual intuition for what an "easy" task looks like.

In [ ]:
easy_df = make_toy_classification_dataset(
    n_samples=300,
    class_sep=1.4,
    flip_y=0.01,
    weights=(0.5, 0.5),
    random_state=42,
)

ax = plot_classification_dataset(easy_df)
ax.set_title("Easy and balanced classification task")
plt.show()

### Discussion prompt

Before fitting any model, ask the audience:

- Which class boundary do you expect a linear model to learn?
- Would accuracy be a reasonable first metric here?
- Which points already look ambiguous?

In [ ]:
easy_results, easy_model = evaluate_logistic_workflow(easy_df)
easy_summary = summarize_classification_results(easy_results, scenario="easy_balanced")
easy_summary.round(3)

## Step 2: Make the task harder without changing the model class

Now we keep the same basic workflow, but we reduce class separation, add some label noise, and introduce class imbalance.

This is an important workshop move: the model family stays the same, but the evaluation problem changes.

In [ ]:
hard_df = make_toy_classification_dataset(
    n_samples=300,
    class_sep=0.7,
    flip_y=0.08,
    weights=(0.8, 0.2),
    random_state=7,
)

ax = plot_classification_dataset(hard_df)
ax.set_title("Harder and imbalanced classification task")
plt.show()

In [ ]:
hard_results, hard_model = evaluate_logistic_workflow(hard_df)
hard_summary = summarize_classification_results(hard_results, scenario="hard_imbalanced")
hard_summary.round(3)

## Step 3: Put the scenarios side by side

The table below is where the pedagogical point becomes clear.
Even when a model still produces a decent accuracy value, the harder setting can degrade balanced accuracy, F1, and confidence around ambiguous cases.

In [ ]:
comparison = pd.concat([easy_summary, hard_summary], ignore_index=True)
comparison.round(3)

- `accuracy` rewards the majority outcome and can look reassuring;
- `balanced_accuracy` reacts more clearly to minority-class failures;
- `f1` becomes more informative when positive predictions are sparse;
- the difficulty proxy tracks how close predicted probabilities are to 0.5, which is a simple way to talk about ambiguous examples.

## Step 4: Inspect hard examples directly

For a hands-on audience, it is often more convincing to inspect specific instances than to discuss metrics abstractly.
Below, we rank test examples by a simple difficulty proxy.

The proxy is high when predicted probabilities are near 0.5, meaning the classifier is uncertain.

In [ ]:
hard_examples = hard_results.sort_values("difficulty_proxy", ascending=False).head(10)
hard_examples[[
    "feature_1",
    "feature_2",
    "label",
    "predicted_label",
    "predicted_probability",
    "difficulty_proxy",
    "correct",
]].round(3)

### Presenter note


> Are these examples hard because the model is weak, or because the items themselves are intrinsically difficult?

Once that question is on the table, IRT becomes a natural next step.

## Step 5: Takeaways

Recommended closing message for this notebook:

1. Standard supervised metrics are useful, but incomplete.
2. Dataset structure changes what those metrics mean.
3. Some examples are systematically harder than others.
4. A latent-ability viewpoint helps separate model skill from item difficulty.

That last point motivates the move to IRT, ICCs, and Beta4.